In [0]:
spark.conf.set(
    "fs.azure.account.auth.type.databrcksgenaistragee.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.databrcksgenaistragee.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.databrcksgenaistragee.dfs.core.windows.net",
    dbutils.secrets.get(scope="databricksprojectscope", key="app-id")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.databrcksgenaistragee.dfs.core.windows.net",
    dbutils.secrets.get(scope="databricksprojectscope", key="app-pwd")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.databrcksgenaistragee.dfs.core.windows.net",
    "https://login.microsoftonline.com/5faad092-ee54-4d69-b6bf-542e30440bd7/oauth2/v2.0/token"
)


In [0]:
from pyspark.sql.functions import input_file_name


bronze_table_path = "testworkspace_7405608515486832.bronze.appointment"

checkpoint_path=f"/Volumes/testworkspace_7405608515486832/bronze/metadata_files/checkpoints/appointment"

container_name ="abfss://raw@databrcksgenaistragee.dfs.core.windows.net/"


if(spark.catalog.tableExists('bronze_table_path')):
    includeExistingFiles_value="False"
else:
    includeExistingFiles_value="True"


# Read data from adlsgen2 container using AutoLoader
df = (
    spark.readStream.format("cloudFiles")
    .option(
        "cloudFiles.format", "csv"
    )  # Change format as needed (csv, json, parquet, etc.)
    .option(
        "cloudFiles.inferSchema", "true"
    )  # Set to "false" if using a predefined schema
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.includeExistingFiles", includeExistingFiles_value)
    .load(container_name)
    .withColumn("source_file", input_file_name())
)

# Write data to Bronze Delta Table in append mode
(
    df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(once=True)
    .toTable(f'{bronze_table_path}')
    .awaitTermination()
    
)
